<a href="https://colab.research.google.com/github/santiagonajera/MODELACION-Y-PRONOSTICOS-DE-LA-DEMANDA/blob/main/ProcesoForecast.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import requests
from io import BytesIO

# URL del archivo Excel
url = "https://github.com/santiagonajera/MODELACION-Y-PRONOSTICOS-DE-LA-DEMANDA/raw/refs/heads/main/EjercicioIntegral-Forecast.xlsx"

# Descargar el archivo
response = requests.get(url)
excel_file = BytesIO(response.content)

# Leer las hojas
forecast_df = pd.read_excel(excel_file, sheet_name='Forecast')
precios_df = pd.read_excel(excel_file, sheet_name='Precios-Costos')

# Limpiar nombres de columnas (eliminar espacios o caracteres extraños)
forecast_df.columns = forecast_df.columns.str.strip()
precios_df.columns = precios_df.columns.str.strip()

# Calcular ventas totales por producto (sumar todos los meses)
forecast_df['Ventas_Totales'] = forecast_df.iloc[:, 1:].sum(axis=1)

# Unir con precios
df_merged = forecast_df.merge(precios_df[['Producto', 'PRECIO']], on='Producto', how='left')

# Convertir PRECIO a numérico (eliminar símbolo $ y espacios)
df_merged['PRECIO'] = pd.to_numeric(df_merged['PRECIO'].str.replace('$', '').str.strip(), errors='coerce')

# Calcular Valor Total de Ventas = Ventas_Totales * PRECIO
df_merged['Valor_Total_Ventas'] = df_merged['Ventas_Totales'] * df_merged['PRECIO']

# Ordenar por Valor_Total_Ventas descendente para clasificación ABC
df_merged = df_merged.sort_values(by='Valor_Total_Ventas', ascending=False).reset_index(drop=True)

# Calcular porcentaje acumulado
total_valor = df_merged['Valor_Total_Ventas'].sum()
df_merged['Porcentaje_Acumulado'] = df_merged['Valor_Total_Ventas'].cumsum() / total_valor * 100

# Asignar clasificación ABC
def classify_abc(porcentaje_acumulado):
    if porcentaje_acumulado <= 60:
        return 'A'
    elif porcentaje_acumulado <= 80:
        return 'B'
    else:
        return 'C'

df_merged['Clasificacion_ABC'] = df_merged['Porcentaje_Acumulado'].apply(classify_abc)

# Mostrar resultados
print("\n=== CLASIFICACIÓN ABC ===")
print(df_merged[['Producto', 'Ventas_Totales', 'PRECIO', 'Valor_Total_Ventas', 'Porcentaje_Acumulado', 'Clasificacion_ABC']])

# Opcional: mostrar resumen por categoría
print("\n=== RESUMEN POR CLASIFICACIÓN ===")
resumen = df_merged.groupby('Clasificacion_ABC').agg(
    Cantidad_Productos=('Producto', 'count'),
    Valor_Total=('Valor_Total_Ventas', 'sum'),
    Porcentaje_del_Total=('Valor_Total_Ventas', lambda x: x.sum() / total_valor * 100)
).round(2)
print(resumen)

AttributeError: Can only use .str accessor with string values!

In [ ]:
import pandas as pd
import requests
from io import BytesIO

# URL del archivo Excel
url = "https://github.com/santiagonajera/MODELACION-Y-PRONOSTICOS-DE-LA-DEMANDA/raw/refs/heads/main/EjercicioIntegral-Forecast.xlsx"

# Descargar el archivo
response = requests.get(url)
excel_file = BytesIO(response.content)

# Leer las hojas
forecast_df = pd.read_excel(excel_file, sheet_name='Forecast')
precios_df = pd.read_excel(excel_file, sheet_name='Precios-Costos')

# Limpiar nombres de columnas
forecast_df.columns = forecast_df.columns.str.strip()
precios_df.columns = precios_df.columns.str.strip()

# Calcular ventas totales por producto
forecast_df['Ventas_Totales'] = forecast_df.iloc[:, 1:].sum(axis=1)

# Unir con precios
df_merged = forecast_df.merge(precios_df[['Producto', 'PRECIO']], on='Producto', how='left')

# --- Conversión segura de PRECIO ---
precio_col = df_merged['PRECIO']

# Si la columna no es de tipo string, asumimos que ya es numérica
if pd.api.types.is_numeric_dtype(precio_col):
    df_merged['PRECIO'] = pd.to_numeric(precio_col, errors='coerce')
else:
    # Si es string, limpiamos símbolos
    df_merged['PRECIO'] = pd.to_numeric(
        precio_col.astype(str).str.replace(r'[$,\s]', '', regex=True),
        errors='coerce'
    )

# Verificar si hay valores nulos en PRECIO
if df_merged['PRECIO'].isnull().any():
    print("⚠️ Advertencia: Algunos precios no pudieron convertirse a número. Se eliminarán esos productos.")
    df_merged = df_merged.dropna(subset=['PRECIO'])

# Calcular Valor Total de Ventas
df_merged['Valor_Total_Ventas'] = df_merged['Ventas_Totales'] * df_merged['PRECIO']

# Ordenar por Valor_Total_Ventas descendente
df_merged = df_merged.sort_values(by='Valor_Total_Ventas', ascending=False).reset_index(drop=True)

# Calcular porcentaje acumulado
total_valor = df_merged['Valor_Total_Ventas'].sum()
df_merged['Porcentaje_Acumulado'] = df_merged['Valor_Total_Ventas'].cumsum() / total_valor * 100

# Asignar clasificación ABC
def classify_abc(porcentaje_acumulado):
    if porcentaje_acumulado <= 60:
        return 'A'
    elif porcentaje_acumulado <= 80:
        return 'B'
    else:
        return 'C'

df_merged['Clasificacion_ABC'] = df_merged['Porcentaje_Acumulado'].apply(classify_abc)

# Mostrar resultados
print("\n=== CLASIFICACIÓN ABC ===")
print(df_merged[['Producto', 'Ventas_Totales', 'PRECIO', 'Valor_Total_Ventas', 'Porcentaje_Acumulado', 'Clasificacion_ABC']])

# Resumen por categoría
print("\n=== RESUMEN POR CLASIFICACIÓN ===")
resumen = df_merged.groupby('Clasificacion_ABC').agg(
    Cantidad_Productos=('Producto', 'count'),
    Valor_Total=('Valor_Total_Ventas', 'sum'),
    Porcentaje_del_Total=('Valor_Total_Ventas', lambda x: x.sum() / total_valor * 100)
).round(2)
print(resumen)


=== CLASIFICACIÓN ABC ===
   Producto  Ventas_Totales  PRECIO  Valor_Total_Ventas  Porcentaje_Acumulado  \
0        P3            5719     5.0             28595.0              4.915681   
1       P23            6437     4.3             27679.1              9.673913   
2        P6            5159     4.7             24247.3             13.842194   
3       P14            5262     4.6             24205.2             18.003238   
4       P20            5505     4.3             23671.5             22.072535   
5       P11            5494     4.3             23624.2             26.133701   
6        P4            5420     4.3             23306.0             30.140166   
7       P13            5801     4.0             23204.0             34.129097   
8       P25            5600     4.1             22960.0             38.076082   
9        P8            5865     3.9             22873.5             42.008197   
10      P19            5259     4.2             22087.8             45.805245   
1

In [ ]:
import pandas as pd
import requests
from io import BytesIO

# URL del archivo
url = "https://github.com/santiagonajera/MODELACION-Y-PRONOSTICOS-DE-LA-DEMANDA/raw/refs/heads/main/EjercicioIntegral-Forecast.xlsx"

# Descargar y leer
response = requests.get(url)
excel_file = BytesIO(response.content)

# Leer la hoja 'Datos'
datos_df = pd.read_excel(excel_file, sheet_name='Datos')

# Limpiar nombres de columnas
datos_df.columns = datos_df.columns.str.strip()

# Identificar columnas de fechas (que no sean 'Producto')
date_cols = [col for col in datos_df.columns if col != 'Producto']

# Filtrar columnas hasta oct-22 inclusive
# Convertimos nombres a formato comparable: 'ene-20' -> (2020, 1), etc.
def parse_period(period):
    try:
        mes, año = period.split('-')
        año = int(año)
        if año < 100:
            año += 2000
        # Mapeo de meses abreviados a número
        meses = {
            'ene': 1, 'feb': 2, 'mar': 3, 'abr': 4,
            'may': 5, 'jun': 6, 'jul': 7, 'ago': 8,
            'sep': 9, 'oct': 10, 'nov': 11, 'dic': 12
        }
        return (año, meses[mes.lower()])
    except:
        return (9999, 99)  # para columnas no válidas

# Crear lista de columnas válidas con su clave de orden
col_info = []
for col in date_cols:
    key = parse_period(col)
    if key != (9999, 99):
        col_info.append((col, key))

# Ordenar por fecha
col_info.sort(key=lambda x: x[1])

# Filtrar hasta oct-22 → (2022, 10)
target_date = (2022, 10)
cols_hasta_oct22 = [col for col, key in col_info if key <= target_date]

if not cols_hasta_oct22:
    raise ValueError("No se encontraron columnas hasta oct-22. Verifica los nombres de las columnas.")

print(f"Usando columnas desde {cols_hasta_oct22[0]} hasta {cols_hasta_oct22[-1]}")

# Calcular media y desviación estándar por producto (solo sobre cols_hasta_oct22)
ventas_historicas = datos_df.set_index('Producto')[cols_hasta_oct22]

# Reemplazar posibles valores no numéricos por NaN
ventas_historicas = ventas_historicas.apply(pd.to_numeric, errors='coerce')

# Calcular media y std
mean_sales = ventas_historicas.mean(axis=1)
std_sales = ventas_historicas.std(axis=1)

# Calcular Coeficiente de Variación (CV)
cv = std_sales / mean_sales
cv = cv.replace([float('inf'), -float('inf')], pd.NA)  # manejar divisiones por cero

# Crear DataFrame de resultados
xyz_df = pd.DataFrame({
    'Producto': mean_sales.index,
    'Media_Ventas': mean_sales.values,
    'Desv_Estandar': std_sales.values,
    'CV': cv.values
}).reset_index(drop=True)

# Eliminar filas con CV faltante
xyz_df = xyz_df.dropna(subset=['CV']).copy()

# Calcular percentiles 33 y 67 del CV
p33 = xyz_df['CV'].quantile(0.33)
p67 = xyz_df['CV'].quantile(0.67)

print(f"\nPercentiles del CV: P33 = {p33:.4f}, P67 = {p67:.4f}")

# Clasificar XYZ (X = más estable = CV bajo)
def classify_xyz(cv_val, p33, p67):
    if cv_val <= p33:
        return 'X'
    elif cv_val <= p67:
        return 'Y'
    else:
        return 'Z'

xyz_df['Clasificacion_XYZ'] = xyz_df['CV'].apply(lambda x: classify_xyz(x, p33, p67))

# Mostrar resultados
print("\n=== CLASIFICACIÓN XYZ (basada en CV de ventas hasta oct-22) ===")
print(xyz_df[['Producto', 'Media_Ventas', 'Desv_Estandar', 'CV', 'Clasificacion_XYZ']].sort_values('CV'))

# Resumen
print("\n=== RESUMEN XYZ ===")
resumen_xyz = xyz_df['Clasificacion_XYZ'].value_counts().sort_index()
print(resumen_xyz)

ValueError: No se encontraron columnas hasta oct-22. Verifica los nombres de las columnas.

In [ ]:
import pandas as pd
import requests
from io import BytesIO

# URL del archivo
url = "https://github.com/santiagonajera/MODELACION-Y-PRONOSTICOS-DE-LA-DEMANDA/raw/refs/heads/main/EjercicioIntegral-Forecast.xlsx"

# Descargar y leer
response = requests.get(url)
excel_file = BytesIO(response.content)

# Leer la hoja 'Datos'
datos_df = pd.read_excel(excel_file, sheet_name='Datos')

# Limpiar nombres de columnas
datos_df.columns = datos_df.columns.str.strip()

# Identificar columnas de períodos (excluyendo 'Producto')
date_cols = [col for col in datos_df.columns if col != 'Producto']

# Mapeo de abreviaturas de meses a número
meses_map = {
    'ene': 1, 'feb': 2, 'mar': 3, 'abr': 4,
    'may': 5, 'jun': 6, 'jul': 7, 'ago': 8,
    'sep': 9, 'oct': 10, 'nov': 11, 'dic': 12
}

def parse_period(period):
    """Convierte 'ene-20' -> (2020, 1), 'oct-25' -> (2025, 10), etc."""
    try:
        parts = period.split('-')
        if len(parts) != 2:
            return None
        mes_str, año_str = parts[0].lower(), parts[1]
        mes = meses_map.get(mes_str)
        if mes is None:
            return None
        año = int(año_str)
        # Asumir que años de 2 dígitos son 20xx (20-29 → 2020-2029)
        if año < 100:
            año += 2000
        return (año, mes)
    except:
        return None

# Extraer y ordenar columnas válidas
col_info = []
for col in date_cols:
    key = parse_period(col)
    if key is not None:
        col_info.append((col, key))

# Ordenar por fecha
col_info.sort(key=lambda x: x[1])

# Definir límite: octubre de 2025 → (2025, 10)
target_date = (2025, 10)
cols_hasta_oct25 = [col for col, key in col_info if key <= target_date]

if not cols_hasta_oct25:
    # Mostrar ejemplos de columnas para diagnóstico
    print("Columnas detectadas en 'Datos':", date_cols[:10])
    print("Ejemplo de parsing:")
    for col in date_cols[:5]:
        print(f"  {col} -> {parse_period(col)}")
    raise ValueError("No se encontraron columnas válidas hasta oct-25. Verifica el formato de los encabezados.")

print(f"✅ Usando datos desde {cols_hasta_oct25[0]} hasta {cols_hasta_oct25[-1]}")

# Extraer datos históricos
ventas_historicas = datos_df.set_index('Producto')[cols_hasta_oct25]

# Asegurar que sean numéricos
ventas_historicas = ventas_historicas.apply(pd.to_numeric, errors='coerce')

# Calcular media y desviación estándar por producto
mean_sales = ventas_historicas.mean(axis=1)
std_sales = ventas_historicas.std(axis=1)

# Calcular Coeficiente de Variación (CV)
cv = std_sales / mean_sales
cv = cv.replace([float('inf'), -float('inf')], pd.NA)

# Crear DataFrame de resultados
xyz_df = pd.DataFrame({
    'Producto': mean_sales.index,
    'Media_Ventas': mean_sales.values,
    'Desv_Estandar': std_sales.values,
    'CV': cv.values
}).reset_index(drop=True)

# Eliminar filas sin CV válido
xyz_df = xyz_df.dropna(subset=['CV']).copy()

if xyz_df.empty:
    raise ValueError("No hay productos con datos válidos para calcular el CV.")

# Calcular percentiles 33% y 67%
p33 = xyz_df['CV'].quantile(0.33)
p67 = xyz_df['CV'].quantile(0.67)

print(f"\n📊 Percentiles del CV: P33 = {p33:.4f}, P67 = {p67:.4f}")

# Clasificar XYZ: X = más estable (bajo CV)
def classify_xyz(cv_val):
    if cv_val <= p33:
        return 'X'
    elif cv_val <= p67:
        return 'Y'
    else:
        return 'Z'

xyz_df['Clasificacion_XYZ'] = xyz_df['CV'].apply(classify_xyz)

# Mostrar resultados
print("\n=== CLASIFICACIÓN XYZ (hasta oct-25) ===")
resultados = xyz_df[['Producto', 'Media_Ventas', 'Desv_Estandar', 'CV', 'Clasificacion_XYZ']].sort_values('CV')
print(resultados.to_string(index=False))

# Resumen
print("\n=== RESUMEN XYZ ===")
resumen = xyz_df['Clasificacion_XYZ'].value_counts().sort_index()
for cat in ['X', 'Y', 'Z']:
    if cat in resumen:
        print(f"{cat}: {resumen[cat]} productos")
    else:
        print(f"{cat}: 0 productos")

Columnas detectadas en 'Datos': [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]
Ejemplo de parsing:
  nan -> None
  nan -> None
  nan -> None
  nan -> None
  nan -> None


ValueError: No se encontraron columnas válidas hasta oct-25. Verifica el formato de los encabezados.

In [ ]:
import pandas as pd
import requests
from io import BytesIO

# URL del archivo
url = "https://github.com/santiagonajera/MODELACION-Y-PRONOSTICOS-DE-LA-DEMANDA/raw/refs/heads/main/EjercicioIntegral-Forecast.xlsx"

# Descargar y leer
response = requests.get(url)
excel_file = BytesIO(response.content)

# Leer la hoja 'Datos' con header=0 (primera fila como encabezados)
datos_df = pd.read_excel(excel_file, sheet_name='Datos', header=0)

# Limpiar nombres de columnas: eliminar espacios, saltos de línea, etc.
datos_df.columns = datos_df.columns.astype(str).str.strip().str.replace('\n', '', regex=False)

# Verificar qué columnas se leyeron
print("✅ Columnas leídas (después de limpieza):")
for i, col in enumerate(datos_df.columns):
    print(f"  [{i}] {repr(col)}")

# Identificar columnas válidas (excluyendo 'Producto')
date_cols = [col for col in datos_df.columns if col != 'Producto']

# Mapeo de meses
meses_map = {
    'ene': 1, 'feb': 2, 'mar': 3, 'abr': 4,
    'may': 5, 'jun': 6, 'jul': 7, 'ago': 8,
    'sep': 9, 'oct': 10, 'nov': 11, 'dic': 12
}

def parse_period(period_str):
    """Convierte 'ene-23' -> (2023, 1), 'oct-25' -> (2025, 10)"""
    try:
        parts = period_str.split('-')
        if len(parts) != 2:
            return None
        mes_str, año_str = parts[0].lower(), parts[1]
        mes = meses_map.get(mes_str)
        if mes is None:
            return None
        año = int(año_str)
        # Asumir que años de 2 dígitos son 20xx (20-29 → 2020-2029)
        if año < 100:
            año += 2000
        return (año, mes)
    except:
        return None

# Filtrar y ordenar columnas válidas
col_info = []
for col in date_cols:
    key = parse_period(col)
    if key is not None:
        col_info.append((col, key))

if not col_info:
    print("\n❌ No se encontraron columnas con formato válido (ej. 'ene-23').")
    print("Verifica que los encabezados estén escritos exactamente como:")
    print("  ene-23, feb-23, ..., oct-25, nov-25, dic-25")
    raise ValueError("No se pudieron identificar columnas de fecha válidas.")

# Ordenar por fecha
col_info.sort(key=lambda x: x[1])

# Definir límite: octubre de 2025 → (2025, 10)
target_date = (2025, 10)
cols_hasta_oct25 = [col for col, key in col_info if key <= target_date]

if not cols_hasta_oct25:
    print(f"\n⚠️ No hay columnas hasta oct-25. Última columna disponible: {col_info[-1][0]}")
    raise ValueError("No se encontraron columnas hasta oct-25.")

print(f"\n✅ Usando {len(cols_hasta_oct25)} columnas desde {cols_hasta_oct25[0]} hasta {cols_hasta_oct25[-1]}")

# Extraer datos históricos
ventas_historicas = datos_df.set_index('Producto')[cols_hasta_oct25]

# Convertir a numérico
ventas_historicas = ventas_historicas.apply(pd.to_numeric, errors='coerce')

# Calcular media y desviación estándar
mean_sales = ventas_historicas.mean(axis=1)
std_sales = ventas_historicas.std(axis=1)

# Calcular CV
cv = std_sales / mean_sales
cv = cv.replace([float('inf'), -float('inf')], pd.NA)

# Crear DataFrame de resultados
xyz_df = pd.DataFrame({
    'Producto': mean_sales.index,
    'Media_Ventas': mean_sales.values,
    'Desv_Estandar': std_sales.values,
    'CV': cv.values
}).reset_index(drop=True)

# Eliminar filas sin CV válido
xyz_df = xyz_df.dropna(subset=['CV']).copy()

if xyz_df.empty:
    raise ValueError("No hay productos con datos válidos para calcular el CV.")

# Calcular percentiles
p33 = xyz_df['CV'].quantile(0.33)
p67 = xyz_df['CV'].quantile(0.67)

print(f"\n📊 Percentiles del CV: P33 = {p33:.4f}, P67 = {p67:.4f}")

# Clasificar XYZ
def classify_xyz(cv_val):
    if cv_val <= p33:
        return 'X'
    elif cv_val <= p67:
        return 'Y'
    else:
        return 'Z'

xyz_df['Clasificacion_XYZ'] = xyz_df['CV'].apply(classify_xyz)

# Mostrar resultados
print("\n=== CLASIFICACIÓN XYZ (hasta oct-25) ===")
resultados = xyz_df[['Producto', 'Media_Ventas', 'Desv_Estandar', 'CV', 'Clasificacion_XYZ']].sort_values('CV')
print(resultados.to_string(index=False))

# Resumen
print("\n=== RESUMEN XYZ ===")
resumen = xyz_df['Clasificacion_XYZ'].value_counts().sort_index()
for cat in ['X', 'Y', 'Z']:
    count = resumen.get(cat, 0)
    print(f"{cat}: {count} productos")

✅ Columnas leídas (después de limpieza):
  [0] 'Producto'
  [1] '2023-01-01 00:00:00'
  [2] '2023-02-01 00:00:00'
  [3] '2023-03-01 00:00:00'
  [4] '2023-04-01 00:00:00'
  [5] '2023-05-01 00:00:00'
  [6] '2023-06-01 00:00:00'
  [7] '2023-07-01 00:00:00'
  [8] '2023-08-01 00:00:00'
  [9] '2023-09-01 00:00:00'
  [10] '2023-10-01 00:00:00'
  [11] '2023-11-01 00:00:00'
  [12] '2023-12-01 00:00:00'
  [13] '2024-01-01 00:00:00'
  [14] '2024-02-01 00:00:00'
  [15] '2024-03-01 00:00:00'
  [16] '2024-04-01 00:00:00'
  [17] '2024-05-01 00:00:00'
  [18] '2024-06-01 00:00:00'
  [19] '2024-07-01 00:00:00'
  [20] '2024-08-01 00:00:00'
  [21] '2024-09-01 00:00:00'
  [22] '2024-10-01 00:00:00'
  [23] '2024-11-01 00:00:00'
  [24] '2024-12-01 00:00:00'
  [25] '2025-01-01 00:00:00'
  [26] '2025-02-01 00:00:00'
  [27] '2025-03-01 00:00:00'
  [28] '2025-04-01 00:00:00'
  [29] '2025-05-01 00:00:00'
  [30] '2025-06-01 00:00:00'
  [31] '2025-07-01 00:00:00'
  [32] '2025-08-01 00:00:00'
  [33] '2025-09-01 00:0

ValueError: No se pudieron identificar columnas de fecha válidas.

In [ ]:
import pandas as pd
import requests
from io import BytesIO

# Descargar archivo
url = "https://github.com/santiagonajera/MODELACION-Y-PRONOSTICOS-DE-LA-DEMANDA/raw/refs/heads/main/EjercicioIntegral-Forecast.xlsx"
response = requests.get(url)
excel_file = BytesIO(response.content)

# Leer hoja 'Datos'
df_datos = pd.read_excel(excel_file, sheet_name='Datos')

# --- Paso 1: Asegurar que la primera columna se llame 'Producto' ---
# Si la primera columna no se llama 'Producto', la renombramos
df_datos.columns = df_datos.columns.astype(str)  # convertir todo a string
df_datos.rename(columns={df_datos.columns[0]: 'Producto'}, inplace=True)

# --- Paso 2: Identificar todas las columnas de período (todas excepto 'Producto') ---
all_cols = df_datos.columns.tolist()
period_cols = [col for col in all_cols if col != 'Producto']

# --- Paso 3: Filtrar manualmente hasta 'oct-25' ---
# Buscar el índice de 'oct-25' (puede estar como 'oct-25', 'Oct-25', etc.)
target_col = None
for col in period_cols:
    if str(col).lower().strip() == 'oct-25':
        target_col = col
        break

if target_col is None:
    # Intentar con variantes comunes
    variants = ['oct-25', 'Oct-25', 'octubre-25', 'Octubre-25', 'oct25', 'Oct25']
    for v in variants:
        if v in period_cols:
            target_col = v
            break

if target_col is None:
    print("❌ No se encontró la columna 'oct-25'. Columnas disponibles:")
    print([col for col in period_cols if 'oct' in str(col).lower()][:10])
    raise ValueError("No se puede encontrar la columna 'oct-25'.")

# Tomar todas las columnas desde el inicio hasta e incluyendo 'oct-25'
idx_target = period_cols.index(target_col)
cols_to_use = period_cols[:idx_target + 1]

print(f"✅ Usando {len(cols_to_use)} columnas: desde {cols_to_use[0]} hasta {cols_to_use[-1]}")

# --- Paso 4: Extraer datos y calcular CV ---
ventas = df_datos[['Producto'] + cols_to_use].set_index('Producto')

# Convertir a numérico (por si hay errores)
ventas = ventas.apply(pd.to_numeric, errors='coerce')

# Calcular media y desviación estándar
mean = ventas.mean(axis=1)
std = ventas.std(axis=1)

# Calcular CV
cv = std / mean
cv = cv.replace([float('inf'), -float('inf')], pd.NA)

# Crear resultado
result = pd.DataFrame({
    'Producto': cv.index,
    'CV': cv.values
}).dropna(subset=['CV']).reset_index(drop=True)

if result.empty:
    raise ValueError("No hay datos válidos para calcular el CV.")

# --- Paso 5: Clasificar XYZ por percentiles ---
p33 = result['CV'].quantile(0.33)
p67 = result['CV'].quantile(0.67)

def xyz_class(cv_val):
    if cv_val <= p33:
        return 'X'
    elif cv_val <= p67:
        return 'Y'
    else:
        return 'Z'

result['XYZ'] = result['CV'].apply(xyz_class)

# --- Mostrar resultados ---
print("\n=== CLASIFICACIÓN XYZ (hasta oct-25) ===")
result_sorted = result.sort_values('CV')
print(result_sorted.to_string(index=False))

print(f"\n📊 Percentiles: P33 = {p33:.4f}, P67 = {p67:.4f}")
print("\n=== RESUMEN ===")
print(result['XYZ'].value_counts().sort_index())


❌ No se encontró la columna 'oct-25'. Columnas disponibles:
[]


ValueError: No se puede encontrar la columna 'oct-25'.

In [ ]:
import pandas as pd
import requests
from io import BytesIO
from datetime import datetime

# Descargar archivo
url = "https://github.com/santiagonajera/MODELACION-Y-PRONOSTICOS-DE-LA-DEMANDA/raw/refs/heads/main/EjercicioIntegral-Forecast.xlsx"
response = requests.get(url)
excel_file = BytesIO(response.content)

# Leer hoja 'Datos'
df_datos = pd.read_excel(excel_file, sheet_name='Datos')

print("=== DIAGNÓSTICO INICIAL ===")
print(f"Dimensiones del DataFrame: {df_datos.shape}")
print(f"\nPrimeras 5 columnas: {df_datos.columns.tolist()[:5]}")
print(f"Últimas 5 columnas: {df_datos.columns.tolist()[-5:]}")

# --- Paso 1: Limpiar nombres de columnas ---
# Convertir columnas a string y limpiar espacios
df_datos.columns = df_datos.columns.astype(str).str.strip()

# Renombrar primera columna a 'Producto'
primera_col = df_datos.columns[0]
df_datos.rename(columns={primera_col: 'Producto'}, inplace=True)

print(f"\n✅ Primera columna renombrada de '{primera_col}' a 'Producto'")

# --- Paso 2: Identificar columnas de período ---
all_cols = df_datos.columns.tolist()
period_cols = [col for col in all_cols if col != 'Producto']

print(f"\n📊 Total de columnas de período: {len(period_cols)}")

# --- Paso 3: Buscar 'oct-25' de forma flexible ---
def find_oct25_column(columns):
    """Busca la columna de octubre 2025 con múltiples estrategias"""

    # Estrategia 1: Buscar exactamente 'oct-25' (case insensitive)
    for col in columns:
        if str(col).lower().strip() == 'oct-25':
            return col, "exacta"

    # Estrategia 2: Buscar variantes comunes
    variants = ['oct-25', 'Oct-25', 'oct 25', 'Oct 25', 'oct25', 'Oct25',
                'octubre-25', 'Octubre-25', 'octubre 25', 'Octubre 25']
    for variant in variants:
        if variant in columns:
            return variant, "variante"

    # Estrategia 3: Buscar como fecha (si Excel lo leyó como timestamp)
    for col in columns:
        col_str = str(col).lower()
        # Puede venir como '2025-10-01' o similar
        if '2025' in col_str and '10' in col_str:
            return col, "timestamp"
        # O como fecha sin formato
        if isinstance(col, datetime):
            if col.year == 2025 and col.month == 10:
                return col, "datetime"

    # Estrategia 4: Buscar por posición (si sabemos cuántas columnas debe haber)
    # Asumiendo que empieza en ene-23 y va hasta oct-25 (34 meses)
    # Esto es una aproximación
    oct_cols = [col for col in columns if 'oct' in str(col).lower()]
    if oct_cols:
        print(f"\n⚠️  Columnas que contienen 'oct': {oct_cols[:5]}")
        # Tomar la última que contenga 'oct' y '25'
        for col in reversed(oct_cols):
            if '25' in str(col):
                return col, "aproximación"

    return None, None

target_col, metodo = find_oct25_column(period_cols)

if target_col is None:
    print("\n❌ ERROR: No se encontró la columna 'oct-25'")
    print(f"\nColumnas disponibles (mostrando primeras 20):")
    for i, col in enumerate(period_cols[:20], 1):
        print(f"  {i}. {col} (tipo: {type(col).__name__})")

    # Mostrar últimas columnas también
    print(f"\nÚltimas 10 columnas:")
    for i, col in enumerate(period_cols[-10:], len(period_cols)-9):
        print(f"  {i}. {col} (tipo: {type(col).__name__})")

    raise ValueError("No se puede encontrar la columna 'oct-25'. Revisa los nombres de columnas arriba.")

print(f"\n✅ Columna encontrada: '{target_col}' (método: {metodo})")

# --- Paso 4: Filtrar hasta oct-25 ---
idx_target = period_cols.index(target_col)
cols_to_use = period_cols[:idx_target + 1]

print(f"\n📅 Usando {len(cols_to_use)} períodos:")
print(f"   Desde: {cols_to_use[0]}")
print(f"   Hasta: {cols_to_use[-1]}")

# --- Paso 5: Extraer datos y calcular CV ---
ventas = df_datos[['Producto'] + cols_to_use].copy()
ventas.set_index('Producto', inplace=True)

# Convertir todo a numérico
for col in ventas.columns:
    ventas[col] = pd.to_numeric(ventas[col], errors='coerce')

# Eliminar filas que tienen todos NaN
ventas = ventas.dropna(how='all')

print(f"\n📊 Productos con datos válidos: {len(ventas)}")

# Calcular estadísticas
mean = ventas.mean(axis=1)
std = ventas.std(axis=1)

# Calcular CV (coeficiente de variación)
# CV = desviación estándar / media
cv = (std / mean).replace([float('inf'), -float('inf')], pd.NA)

# Crear DataFrame de resultados
result = pd.DataFrame({
    'Producto': cv.index,
    'Media': mean.values,
    'Desv_Std': std.values,
    'CV': cv.values
}).dropna(subset=['CV']).reset_index(drop=True)

if result.empty:
    raise ValueError("❌ No hay datos válidos para calcular el CV.")

print(f"✅ CV calculado para {len(result)} productos")

# --- Paso 6: Clasificar XYZ por percentiles ---
p33 = result['CV'].quantile(0.33)
p67 = result['CV'].quantile(0.67)

def xyz_class(cv_val):
    """Clasifica según percentiles del CV"""
    if cv_val <= p33:
        return 'X'  # Demanda más estable
    elif cv_val <= p67:
        return 'Y'  # Demanda moderada
    else:
        return 'Z'  # Demanda más variable

result['XYZ'] = result['CV'].apply(xyz_class)

# --- Paso 7: Mostrar resultados ---
print("\n" + "="*70)
print("CLASIFICACIÓN XYZ (Análisis de Variabilidad)")
print(f"Período: {cols_to_use[0]} hasta {cols_to_use[-1]}")
print("="*70)

result_sorted = result.sort_values('CV').reset_index(drop=True)

print("\n📋 TOP 10 productos más estables (menor CV):")
print(result_sorted[['Producto', 'CV', 'XYZ']].head(10).to_string(index=False))

print("\n📋 TOP 10 productos más variables (mayor CV):")
print(result_sorted[['Producto', 'CV', 'XYZ']].tail(10).to_string(index=False))

print(f"\n📊 PERCENTILES:")
print(f"   P33 = {p33:.4f} (hasta aquí es clase X)")
print(f"   P67 = {p67:.4f} (hasta aquí es clase Y)")

print("\n📊 RESUMEN POR CLASE:")
resumen = result['XYZ'].value_counts().sort_index()
for clase in ['X', 'Y', 'Z']:
    if clase in resumen.index:
        cant = resumen[clase]
        porc = (cant / len(result)) * 100
        print(f"   Clase {clase}: {cant:3d} productos ({porc:5.1f}%)")

# --- Paso 8: Exportar resultados ---
output_file = 'clasificacion_xyz_resultados.xlsx'
with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    result_sorted.to_excel(writer, sheet_name='Clasificacion XYZ', index=False)

    # Crear resumen
    resumen_df = pd.DataFrame({
        'Clase': ['X', 'Y', 'Z'],
        'Cantidad': [resumen.get('X', 0), resumen.get('Y', 0), resumen.get('Z', 0)],
        'Porcentaje': [
            (resumen.get('X', 0) / len(result)) * 100,
            (resumen.get('Y', 0) / len(result)) * 100,
            (resumen.get('Z', 0) / len(result)) * 100
        ],
        'Descripción': [
            'Demanda estable (CV bajo)',
            'Demanda moderada (CV medio)',
            'Demanda variable (CV alto)'
        ]
    })
    resumen_df.to_excel(writer, sheet_name='Resumen', index=False)

print(f"\n✅ Resultados exportados a: {output_file}")
print("\n" + "="*70)

=== DIAGNÓSTICO INICIAL ===
Dimensiones del DataFrame: (30, 49)

Primeras 5 columnas: ['Producto', datetime.datetime(2023, 1, 1, 0, 0), datetime.datetime(2023, 2, 1, 0, 0), datetime.datetime(2023, 3, 1, 0, 0), datetime.datetime(2023, 4, 1, 0, 0)]
Últimas 5 columnas: [datetime.datetime(2026, 8, 1, 0, 0), datetime.datetime(2026, 9, 1, 0, 0), datetime.datetime(2026, 10, 1, 0, 0), datetime.datetime(2026, 11, 1, 0, 0), datetime.datetime(2026, 12, 1, 0, 0)]

✅ Primera columna renombrada de 'Producto' a 'Producto'

📊 Total de columnas de período: 48

✅ Columna encontrada: '2025-10-01 00:00:00' (método: timestamp)

📅 Usando 34 períodos:
   Desde: 2023-01-01 00:00:00
   Hasta: 2025-10-01 00:00:00

📊 Productos con datos válidos: 30
✅ CV calculado para 30 productos

CLASIFICACIÓN XYZ (Análisis de Variabilidad)
Período: 2023-01-01 00:00:00 hasta 2025-10-01 00:00:00

📋 TOP 10 productos más estables (menor CV):
Producto       CV XYZ
      P9 0.522667   X
     P19 0.523071   X
     P16 0.543759   X
 

In [ ]:
import pandas as pd
import numpy as np
import requests
from io import BytesIO

print("="*80)
print("CLASIFICACIÓN DE DEMANDA: INTERMITENTE vs CONTINUA (ADI)")
print("="*80)

# --- Paso 1: Descargar y leer datos ---
url = "https://github.com/santiagonajera/MODELACION-Y-PRONOSTICOS-DE-LA-DEMANDA/raw/refs/heads/main/EjercicioIntegral-Forecast.xlsx"
response = requests.get(url)
excel_file = BytesIO(response.content)

df_datos = pd.read_excel(excel_file, sheet_name='Datos')

# Limpiar nombres de columnas
df_datos.columns = df_datos.columns.astype(str).str.strip()
df_datos.rename(columns={df_datos.columns[0]: 'Producto'}, inplace=True)

# --- Paso 2: Buscar columna 'oct-25' ---
all_cols = df_datos.columns.tolist()
period_cols = [col for col in all_cols if col != 'Producto']

def find_oct25_column(columns):
    """Busca la columna de octubre 2025"""
    for col in columns:
        if str(col).lower().strip() == 'oct-25':
            return col

    variants = ['oct-25', 'Oct-25', 'oct 25', 'Oct 25', 'oct25', 'Oct25']
    for variant in variants:
        if variant in columns:
            return variant

    # Buscar por timestamp
    for col in columns:
        col_str = str(col).lower()
        if '2025' in col_str and '10' in col_str:
            return col

    # Buscar columnas con 'oct' y '25'
    oct_cols = [col for col in columns if 'oct' in str(col).lower() and '25' in str(col)]
    if oct_cols:
        return oct_cols[0]

    return None

target_col = find_oct25_column(period_cols)
if target_col is None:
    raise ValueError("No se encontró la columna 'oct-25'")

idx_target = period_cols.index(target_col)
cols_to_use = period_cols[:idx_target + 1]

print(f"\n✅ Periodo de análisis: {cols_to_use[0]} hasta {cols_to_use[-1]}")
print(f"📊 Total de períodos: {len(cols_to_use)}")

# --- Paso 3: Preparar datos para análisis ---
ventas = df_datos[['Producto'] + cols_to_use].copy()
ventas.set_index('Producto', inplace=True)

# Convertir a numérico
for col in ventas.columns:
    ventas[col] = pd.to_numeric(ventas[col], errors='coerce')

ventas = ventas.dropna(how='all')

# --- Paso 4: Calcular clasificación XYZ (CV) ---
print("\n" + "="*80)
print("PASO 1: CALCULANDO CLASIFICACIÓN XYZ")
print("="*80)

mean = ventas.mean(axis=1)
std = ventas.std(axis=1)
cv = (std / mean).replace([float('inf'), -float('inf')], pd.NA)

result_xyz = pd.DataFrame({
    'Producto': cv.index,
    'Media': mean.values,
    'Desv_Std': std.values,
    'CV': cv.values
}).dropna(subset=['CV']).reset_index(drop=True)

# Clasificar XYZ
p33 = result_xyz['CV'].quantile(0.33)
p67 = result_xyz['CV'].quantile(0.67)

def xyz_class(cv_val):
    if cv_val <= p33:
        return 'X'
    elif cv_val <= p67:
        return 'Y'
    else:
        return 'Z'

result_xyz['XYZ'] = result_xyz['CV'].apply(xyz_class)

print(f"\n📊 Clasificación XYZ completada:")
print(result_xyz['XYZ'].value_counts().sort_index())

# --- Paso 5: Calcular ADI para items Z ---
print("\n" + "="*80)
print("PASO 2: CALCULANDO ADI PARA ITEMS CLASE Z")
print("="*80)

items_z = result_xyz[result_xyz['XYZ'] == 'Z']['Producto'].tolist()
print(f"\n📋 Items clasificados como Z: {len(items_z)}")

def calcular_adi(serie):
    """
    Calcula el Average Demand Interval (ADI)
    ADI = Número total de períodos / Número de períodos con demanda > 0

    ADI < 1.32: Demanda continua/suave
    ADI >= 1.32: Demanda intermitente
    """
    total_periodos = len(serie)
    periodos_con_demanda = (serie > 0).sum()

    if periodos_con_demanda == 0:
        return np.nan

    adi = total_periodos / periodos_con_demanda
    return adi

# Calcular ADI solo para items Z
adi_results = []

for producto in items_z:
    if producto in ventas.index:
        serie = ventas.loc[producto]
        adi = calcular_adi(serie)

        # Clasificar según ADI
        if pd.isna(adi):
            tipo_demanda = 'Sin datos'
        elif adi < 1.32:
            tipo_demanda = 'Continua'
        else:
            tipo_demanda = 'Intermitente'

        # Estadísticas adicionales
        total_periodos = len(serie)
        periodos_con_demanda = (serie > 0).sum()
        periodos_sin_demanda = total_periodos - periodos_con_demanda

        adi_results.append({
            'Producto': producto,
            'ADI': adi,
            'Tipo_Demanda': tipo_demanda,
            'Total_Periodos': total_periodos,
            'Periodos_Con_Demanda': periodos_con_demanda,
            'Periodos_Sin_Demanda': periodos_sin_demanda,
            'Porc_Periodos_Activos': (periodos_con_demanda / total_periodos) * 100
        })

df_adi = pd.DataFrame(adi_results)

print(f"\n✅ ADI calculado para {len(df_adi)} items Z")
print(f"\n📊 Clasificación de items Z por tipo de demanda:")
print(df_adi['Tipo_Demanda'].value_counts())

# --- Paso 6: Crear matriz completa de clasificación ---
print("\n" + "="*80)
print("PASO 3: GENERANDO MATRIZ COMPLETA DE CLASIFICACIÓN")
print("="*80)

# Combinar XYZ con ADI
matriz_completa = result_xyz[['Producto', 'CV', 'XYZ']].copy()

# Agregar columnas de ADI
matriz_completa = matriz_completa.merge(
    df_adi[['Producto', 'ADI', 'Tipo_Demanda', 'Periodos_Con_Demanda', 'Porc_Periodos_Activos']],
    on='Producto',
    how='left'
)

# Para items X e Y, asignar tipo de demanda "Continua"
matriz_completa['Tipo_Demanda'] = matriz_completa.apply(
    lambda row: row['Tipo_Demanda'] if row['XYZ'] == 'Z' else 'Continua',
    axis=1
)

# Crear clasificación combinada
def clasificacion_combinada(row):
    if row['XYZ'] in ['X', 'Y']:
        return f"{row['XYZ']}-Continua"
    else:  # Z
        return f"Z-{row['Tipo_Demanda']}"

matriz_completa['Clasificacion_Final'] = matriz_completa.apply(clasificacion_combinada, axis=1)

# Ordenar por clasificación
matriz_completa = matriz_completa.sort_values(['XYZ', 'Tipo_Demanda', 'CV'])

print("\n✅ Matriz completa generada")
print(f"\n📊 RESUMEN FINAL DE CLASIFICACIÓN:")
print(matriz_completa['Clasificacion_Final'].value_counts().sort_index())

# --- Paso 7: Mostrar resultados detallados ---
print("\n" + "="*80)
print("RESULTADOS DETALLADOS")
print("="*80)

print("\n📌 ITEMS CLASE X (Demanda estable y continua):")
items_x = matriz_completa[matriz_completa['XYZ'] == 'X']
print(f"   Total: {len(items_x)} items")

print("\n📌 ITEMS CLASE Y (Demanda moderada y continua):")
items_y = matriz_completa[matriz_completa['XYZ'] == 'Y']
print(f"   Total: {len(items_y)} items")

print("\n📌 ITEMS CLASE Z (Demanda variable):")
items_z_continua = matriz_completa[matriz_completa['Clasificacion_Final'] == 'Z-Continua']
items_z_intermitente = matriz_completa[matriz_completa['Clasificacion_Final'] == 'Z-Intermitente']

print(f"\n   Z-Continua (ADI < 1.32): {len(items_z_continua)} items")
if len(items_z_continua) > 0:
    print(f"   • ADI promedio: {items_z_continua['ADI'].mean():.3f}")
    print(f"   • % períodos activos: {items_z_continua['Porc_Periodos_Activos'].mean():.1f}%")
    print("\n   Top 5 items Z-Continua:")
    print(items_z_continua[['Producto', 'CV', 'ADI', 'Porc_Periodos_Activos']].head().to_string(index=False))

print(f"\n   Z-Intermitente (ADI >= 1.32): {len(items_z_intermitente)} items")
if len(items_z_intermitente) > 0:
    print(f"   • ADI promedio: {items_z_intermitente['ADI'].mean():.3f}")
    print(f"   • % períodos activos: {items_z_intermitente['Porc_Periodos_Activos'].mean():.1f}%")
    print("\n   Top 5 items Z-Intermitente:")
    print(items_z_intermitente[['Producto', 'CV', 'ADI', 'Porc_Periodos_Activos']].head().to_string(index=False))

# --- Paso 8: Crear resumen estadístico ---
print("\n" + "="*80)
print("ESTADÍSTICAS POR TIPO DE DEMANDA")
print("="*80)

resumen_stats = matriz_completa.groupby('Clasificacion_Final').agg({
    'Producto': 'count',
    'CV': ['mean', 'min', 'max'],
    'ADI': ['mean', 'min', 'max']
}).round(3)

resumen_stats.columns = ['Cantidad', 'CV_Prom', 'CV_Min', 'CV_Max', 'ADI_Prom', 'ADI_Min', 'ADI_Max']
print("\n" + resumen_stats.to_string())

# --- Paso 9: Exportar resultados ---
print("\n" + "="*80)
print("EXPORTANDO RESULTADOS")
print("="*80)

output_file = 'clasificacion_demanda_ADI.xlsx'

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    # Hoja 1: Matriz completa
    matriz_completa.to_excel(writer, sheet_name='Matriz_Completa', index=False)

    # Hoja 2: Solo items Z con ADI
    df_adi_sorted = df_adi.sort_values('ADI', ascending=False)
    df_adi_sorted.to_excel(writer, sheet_name='Items_Z_ADI', index=False)

    # Hoja 3: Resumen por clasificación
    resumen = matriz_completa['Clasificacion_Final'].value_counts().reset_index()
    resumen.columns = ['Clasificacion', 'Cantidad']
    resumen['Porcentaje'] = (resumen['Cantidad'] / len(matriz_completa) * 100).round(2)
    resumen.to_excel(writer, sheet_name='Resumen', index=False)

    # Hoja 4: Items continua vs intermitente (solo Z)
    tipo_demanda = df_adi['Tipo_Demanda'].value_counts().reset_index()
    tipo_demanda.columns = ['Tipo_Demanda', 'Cantidad']
    tipo_demanda['Porcentaje'] = (tipo_demanda['Cantidad'] / len(df_adi) * 100).round(2)
    tipo_demanda.to_excel(writer, sheet_name='Z_Continua_vs_Intermitente', index=False)

    # Hoja 5: Estadísticas detalladas
    resumen_stats.to_excel(writer, sheet_name='Estadisticas')

print(f"\n✅ Resultados exportados a: {output_file}")

# --- Paso 10: Crear matriz visual simple ---
print("\n" + "="*80)
print("MATRIZ VISUAL DE CLASIFICACIÓN")
print("="*80)

matriz_visual = pd.DataFrame({
    'Clasificacion': ['X-Continua', 'Y-Continua', 'Z-Continua', 'Z-Intermitente'],
    'Cantidad': [
        len(items_x),
        len(items_y),
        len(items_z_continua),
        len(items_z_intermitente)
    ],
    'Descripcion': [
        'Demanda estable y predecible',
        'Demanda moderada y predecible',
        'Demanda variable pero continua (ADI < 1.32)',
        'Demanda esporádica (ADI >= 1.32)'
    ],
    'Estrategia_Sugerida': [
        'Pronósticos simples, EOQ, stock bajo',
        'Pronósticos regulares, stock medio',
        'Modelos avanzados, análisis frecuente',
        'Croston, SBA, gestión por excepción'
    ]
})

print("\n" + matriz_visual.to_string(index=False))

print("\n" + "="*80)
print("PROCESO COMPLETADO")
print("="*80)

print(f"""
✅ Resumen Final:
   • Total de items analizados: {len(matriz_completa)}
   • Items X (continua): {len(items_x)} ({len(items_x)/len(matriz_completa)*100:.1f}%)
   • Items Y (continua): {len(items_y)} ({len(items_y)/len(matriz_completa)*100:.1f}%)
   • Items Z-Continua: {len(items_z_continua)} ({len(items_z_continua)/len(matriz_completa)*100:.1f}%)
   • Items Z-Intermitente: {len(items_z_intermitente)} ({len(items_z_intermitente)/len(matriz_completa)*100:.1f}%)

📊 Umbral ADI usado: 1.32
   • ADI < 1.32 → Demanda Continua
   • ADI >= 1.32 → Demanda Intermitente

📁 Archivo generado: {output_file}
""")

CLASIFICACIÓN DE DEMANDA: INTERMITENTE vs CONTINUA (ADI)

✅ Periodo de análisis: 2023-01-01 00:00:00 hasta 2025-10-01 00:00:00
📊 Total de períodos: 34

PASO 1: CALCULANDO CLASIFICACIÓN XYZ

📊 Clasificación XYZ completada:
XYZ
X    10
Y    10
Z    10
Name: count, dtype: int64

PASO 2: CALCULANDO ADI PARA ITEMS CLASE Z

📋 Items clasificados como Z: 10

✅ ADI calculado para 10 items Z

📊 Clasificación de items Z por tipo de demanda:
Tipo_Demanda
Continua        7
Intermitente    3
Name: count, dtype: int64

PASO 3: GENERANDO MATRIZ COMPLETA DE CLASIFICACIÓN

✅ Matriz completa generada

📊 RESUMEN FINAL DE CLASIFICACIÓN:
Clasificacion_Final
X-Continua        10
Y-Continua        10
Z-Continua         7
Z-Intermitente     3
Name: count, dtype: int64

RESULTADOS DETALLADOS

📌 ITEMS CLASE X (Demanda estable y continua):
   Total: 10 items

📌 ITEMS CLASE Y (Demanda moderada y continua):
   Total: 10 items

📌 ITEMS CLASE Z (Demanda variable):

   Z-Continua (ADI < 1.32): 7 items
   • ADI promedi

In [ ]:
import pandas as pd
import numpy as np
import requests
from io import BytesIO

print("="*80)
print("CLASIFICACIÓN COMPLETA: ABC-XYZ-ADI-CV²")
print("="*80)

# --- Paso 1: Descargar y leer datos ---
url = "https://github.com/santiagonajera/MODELACION-Y-PRONOSTICOS-DE-LA-DEMANDA/raw/refs/heads/main/EjercicioIntegral-Forecast.xlsx"
response = requests.get(url)
excel_file = BytesIO(response.content)

df_datos = pd.read_excel(excel_file, sheet_name='Datos')

# Limpiar nombres de columnas
df_datos.columns = df_datos.columns.astype(str).str.strip()
df_datos.rename(columns={df_datos.columns[0]: 'Producto'}, inplace=True)

# --- Paso 2: Buscar columna 'oct-25' ---
all_cols = df_datos.columns.tolist()
period_cols = [col for col in all_cols if col != 'Producto']

def find_oct25_column(columns):
    """Busca la columna de octubre 2025"""
    for col in columns:
        if str(col).lower().strip() == 'oct-25':
            return col

    variants = ['oct-25', 'Oct-25', 'oct 25', 'Oct 25', 'oct25', 'Oct25']
    for variant in variants:
        if variant in columns:
            return variant

    for col in columns:
        col_str = str(col).lower()
        if '2025' in col_str and '10' in col_str:
            return col

    oct_cols = [col for col in columns if 'oct' in str(col).lower() and '25' in str(col)]
    if oct_cols:
        return oct_cols[0]

    return None

target_col = find_oct25_column(period_cols)
if target_col is None:
    raise ValueError("No se encontró la columna 'oct-25'")

idx_target = period_cols.index(target_col)
cols_to_use = period_cols[:idx_target + 1]

print(f"\n✅ Periodo: {cols_to_use[0]} hasta {cols_to_use[-1]} ({len(cols_to_use)} períodos)")

# --- Paso 3: Preparar datos ---
ventas = df_datos[['Producto'] + cols_to_use].copy()
ventas.set_index('Producto', inplace=True)

# Convertir a numérico
for col in ventas.columns:
    ventas[col] = pd.to_numeric(ventas[col], errors='coerce')

ventas = ventas.dropna(how='all')

print(f"📊 Productos analizados: {len(ventas)}")

# --- Paso 4: Clasificación ABC ---
print("\n" + "="*80)
print("PASO 1: CLASIFICACIÓN ABC")
print("="*80)

# Calcular ventas totales
ventas_totales = ventas.sum(axis=1).sort_values(ascending=False)
ventas_totales_pct = (ventas_totales / ventas_totales.sum() * 100)
ventas_acum_pct = ventas_totales_pct.cumsum()

# Clasificar ABC
def clasificar_abc(pct_acum):
    if pct_acum <= 80:
        return 'A'
    elif pct_acum <= 95:
        return 'B'
    else:
        return 'C'

abc_class = ventas_acum_pct.apply(clasificar_abc)

print(f"\n📊 Clasificación ABC (80-15-5):")
print(abc_class.value_counts().sort_index())

# --- Paso 5: Clasificación XYZ (CV) ---
print("\n" + "="*80)
print("PASO 2: CLASIFICACIÓN XYZ (Coeficiente de Variación)")
print("="*80)

mean = ventas.mean(axis=1)
std = ventas.std(axis=1)
cv = (std / mean).replace([float('inf'), -float('inf')], pd.NA)

# Clasificar XYZ por percentiles
p33 = cv.quantile(0.33)
p67 = cv.quantile(0.67)

def xyz_class(cv_val):
    if pd.isna(cv_val):
        return None
    if cv_val <= p33:
        return 'X'
    elif cv_val <= p67:
        return 'Y'
    else:
        return 'Z'

xyz = cv.apply(xyz_class)

print(f"\n📊 Clasificación XYZ:")
print(xyz.value_counts().sort_index())
print(f"   P33 = {p33:.4f}")
print(f"   P67 = {p67:.4f}")

# --- Paso 6: Calcular ADI para items Z ---
print("\n" + "="*80)
print("PASO 3: CÁLCULO DE ADI (Average Demand Interval) para items Z")
print("="*80)

def calcular_adi(serie):
    """
    ADI = Total períodos / Períodos con demanda > 0
    ADI < 1.32: Continua
    ADI >= 1.32: Intermitente
    """
    total_periodos = len(serie)
    periodos_con_demanda = (serie > 0).sum()

    if periodos_con_demanda == 0:
        return np.nan

    return total_periodos / periodos_con_demanda

# Calcular ADI para todos
adi = ventas.apply(calcular_adi, axis=1)

# Clasificar tipo de demanda
def tipo_demanda(row):
    if row['XYZ'] in ['X', 'Y']:
        return 'Continua'
    else:  # Z
        if pd.isna(row['ADI']):
            return 'Sin datos'
        elif row['ADI'] < 1.32:
            return 'Continua'
        else:
            return 'Intermitente'

# Crear DataFrame maestro
df_master = pd.DataFrame({
    'Producto': ventas.index,
    'Ventas_Total': ventas_totales,
    'ABC': abc_class,
    'Media': mean,
    'Desv_Std': std,
    'CV': cv,
    'XYZ': xyz,
    'ADI': adi
})

df_master['Tipo_Demanda'] = df_master.apply(tipo_demanda, axis=1)

print(f"\n📊 Tipo de Demanda:")
print(df_master['Tipo_Demanda'].value_counts())

# --- Paso 7: Calcular CV² para items intermitentes ---
print("\n" + "="*80)
print("PASO 4: CÁLCULO DE CV² para items INTERMITENTES")
print("="*80)

print("""
📖 Clasificación según CV² (Syntetos et al.):
   • CV² < 0.49: Suave (Smooth) - Forecasteable con Croston/SBA
   • CV² >= 0.49: Errática (Erratic) - Muy difícil de forecastear
""")

# Calcular CV² para todos (aunque solo nos interesan los intermitentes)
df_master['CV2'] = df_master['CV'] ** 2

# Clasificar según CV² (solo para intermitentes)
def clasificar_cv2(row):
    if row['Tipo_Demanda'] == 'Intermitente':
        if pd.isna(row['CV2']):
            return 'Sin datos'
        elif row['CV2'] < 0.49:
            return 'Suave (Forecasteable)'
        else:
            return 'Errática (Difícil)'
    else:
        return 'N/A'

df_master['CV2_Clasificacion'] = df_master.apply(clasificar_cv2, axis=1)

# Mostrar estadísticas de items intermitentes
items_intermitentes = df_master[df_master['Tipo_Demanda'] == 'Intermitente']

if len(items_intermitentes) > 0:
    print(f"\n📊 Total items intermitentes: {len(items_intermitentes)}")
    print(f"\n   Estadísticas de CV²:")
    print(f"   • Media: {items_intermitentes['CV2'].mean():.4f}")
    print(f"   • Mediana: {items_intermitentes['CV2'].median():.4f}")
    print(f"   • Mínimo: {items_intermitentes['CV2'].min():.4f}")
    print(f"   • Máximo: {items_intermitentes['CV2'].max():.4f}")

    print(f"\n📊 Clasificación por CV²:")
    cv2_counts = df_master[df_master['Tipo_Demanda'] == 'Intermitente']['CV2_Clasificacion'].value_counts()
    print(cv2_counts)

    print(f"\n🔍 Detalle de items intermitentes:")
    print(items_intermitentes[['Producto', 'ABC', 'CV', 'CV2', 'ADI', 'CV2_Clasificacion']].sort_values('CV2', ascending=False).to_string(index=False))
else:
    print("⚠️  No hay items intermitentes en este dataset")

# --- Paso 8: Clasificación Final Detallada ---
print("\n" + "="*80)
print("PASO 5: CLASIFICACIÓN FINAL DETALLADA")
print("="*80)

def clasificacion_final_detallada(row):
    """
    Genera la clasificación final completa
    """
    abc = row['ABC']
    xyz = row['XYZ']
    tipo = row['Tipo_Demanda']

    if tipo == 'Continua':
        return f"{abc}{xyz}-Continua"
    elif tipo == 'Intermitente':
        cv2_clase = row['CV2_Clasificacion']
        if 'Suave' in cv2_clase:
            return f"{abc}{xyz}-Intermitente-Suave"
        elif 'Errática' in cv2_clase:
            return f"{abc}{xyz}-Intermitente-Errática"
        else:
            return f"{abc}{xyz}-Intermitente-{cv2_clase}"
    else:
        return f"{abc}{xyz}-{tipo}"

df_master['Clasificacion_Final'] = df_master.apply(clasificacion_final_detallada, axis=1)

# --- Paso 9: GENERAR DATAFRAME RESUMEN SOLICITADO ---
print("\n" + "="*80)
print("📊 DATAFRAME RESUMEN: CONTEO POR COMBINACIÓN ABC-XYZ-TIPO")
print("="*80)

# Crear DataFrame de resumen con todas las combinaciones
resumen = df_master.groupby(['ABC', 'XYZ', 'Tipo_Demanda', 'CV2_Clasificacion']).size().reset_index(name='Cantidad')

# Crear columna de clasificación simplificada
resumen['Clasificacion'] = resumen.apply(
    lambda row: f"{row['ABC']}{row['XYZ']}-" +
                (f"{row['Tipo_Demanda']}" if row['Tipo_Demanda'] == 'Continua'
                 else f"Intermitente ({row['CV2_Clasificacion']})"),
    axis=1
)

# Ordenar
resumen = resumen.sort_values(['ABC', 'XYZ', 'Tipo_Demanda'])

print("\n📋 RESUMEN DETALLADO:")
print(resumen.to_string(index=False))

# Crear DataFrame resumen simplificado
print("\n" + "="*80)
print("📊 RESUMEN SIMPLIFICADO (Agrupado)")
print("="*80)

# Agrupar por ABC-XYZ-Tipo (sin distinguir CV2 para continuas)
resumen_simple = df_master.groupby(['ABC', 'XYZ', 'Tipo_Demanda']).agg({
    'Producto': 'count',
    'Ventas_Total': 'sum',
    'CV': 'mean',
    'ADI': 'mean',
    'CV2': 'mean'
}).reset_index()

resumen_simple.columns = ['ABC', 'XYZ', 'Tipo_Demanda', 'Cantidad', 'Ventas_Total', 'CV_Promedio', 'ADI_Promedio', 'CV2_Promedio']

# Crear columna combinada
resumen_simple['Grupo'] = resumen_simple['ABC'] + resumen_simple['XYZ'] + '-' + resumen_simple['Tipo_Demanda']

# Redondear valores
resumen_simple['CV_Promedio'] = resumen_simple['CV_Promedio'].round(4)
resumen_simple['ADI_Promedio'] = resumen_simple['ADI_Promedio'].round(3)
resumen_simple['CV2_Promedio'] = resumen_simple['CV2_Promedio'].round(4)
resumen_simple['Ventas_Total'] = resumen_simple['Ventas_Total'].round(0)

print("\n" + resumen_simple.to_string(index=False))

# --- Paso 10: Resumen por combinación ABC-XYZ-TIPO ---
print("\n" + "="*80)
print("📊 TABLA FINAL: CANTIDAD POR GRUPO")
print("="*80)

# Crear tabla pivote
tabla_final = resumen_simple[['Grupo', 'Cantidad', 'CV_Promedio', 'ADI_Promedio']].copy()
tabla_final = tabla_final.sort_values('Cantidad', ascending=False)

print("\n" + tabla_final.to_string(index=False))

# Calcular totales y porcentajes
total_items = len(df_master)

print("\n" + "="*80)
print("📊 DISTRIBUCIÓN PORCENTUAL")
print("="*80)

tabla_pct = tabla_final.copy()
tabla_pct['Porcentaje'] = (tabla_pct['Cantidad'] / total_items * 100).round(2)
tabla_pct['Acumulado'] = tabla_pct['Porcentaje'].cumsum().round(2)

print("\n" + tabla_pct.to_string(index=False))

# --- Paso 11: Análisis especial de intermitentes ---
if len(items_intermitentes) > 0:
    print("\n" + "="*80)
    print("🚨 ANÁLISIS DETALLADO DE ITEMS INTERMITENTES")
    print("="*80)

    # Resumen por ABC para intermitentes
    intermitentes_abc = items_intermitentes.groupby(['ABC', 'CV2_Clasificacion']).size().reset_index(name='Cantidad')
    print("\n📊 Items Intermitentes por clase ABC y CV²:")
    print(intermitentes_abc.to_string(index=False))

    # Estadísticas por tipo
    print("\n📊 Estadísticas de Items Intermitentes:")

    suaves = items_intermitentes[items_intermitentes['CV2_Clasificacion'] == 'Suave (Forecasteable)']
    erraticas = items_intermitentes[items_intermitentes['CV2_Clasificacion'] == 'Errática (Difícil)']

    if len(suaves) > 0:
        print(f"\n   🟢 Intermitentes SUAVES (CV² < 0.49): {len(suaves)} items")
        print(f"      • Representan: {len(suaves)/len(items_intermitentes)*100:.1f}% de intermitentes")
        print(f"      • CV² promedio: {suaves['CV2'].mean():.4f}")
        print(f"      • ADI promedio: {suaves['ADI'].mean():.3f}")
        print(f"      • Método sugerido: Croston, SBA, TSB")

    if len(erraticas) > 0:
        print(f"\n   🔴 Intermitentes ERRÁTICAS (CV² >= 0.49): {len(erraticas)} items")
        print(f"      • Representan: {len(erraticas)/len(items_intermitentes)*100:.1f}% de intermitentes")
        print(f"      • CV² promedio: {erraticas['CV2'].mean():.4f}")
        print(f"      • ADI promedio: {erraticas['ADI'].mean():.3f}")
        print(f"      • Método sugerido: Bootstrap, Simulación, Gestión por excepción")

# --- Paso 12: Matriz de decisión ---
print("\n" + "="*80)
print("🎯 MATRIZ DE DECISIÓN Y ESTRATEGIAS")
print("="*80)

matriz_decision = []

for _, row in resumen_simple.iterrows():
    grupo = row['Grupo']
    cant = row['Cantidad']
    abc = row['ABC']
    xyz = row['XYZ']
    tipo = row['Tipo_Demanda']

    # Determinar prioridad
    if abc == 'A':
        prioridad = 'ALTA'
    elif abc == 'B':
        prioridad = 'MEDIA'
    else:
        prioridad = 'BAJA'

    # Determinar método de pronóstico
    if tipo == 'Continua':
        if xyz == 'X':
            metodo = 'Promedio Móvil, SES'
        elif xyz == 'Y':
            metodo = 'Holt, Winter\'s'
        else:  # Z
            metodo = 'ARIMA, ML'
    else:  # Intermitente
        cv2_prom = row['CV2_Promedio']
        if pd.notna(cv2_prom) and cv2_prom < 0.49:
            metodo = 'Croston, SBA'
        else:
            metodo = 'TSB, Bootstrap'

    # Determinar estrategia de inventario
    if tipo == 'Continua':
        if xyz == 'X':
            inventario = 'Stock bajo, EOQ'
        elif xyz == 'Y':
            inventario = 'Stock medio, (s,Q)'
        else:
            inventario = 'Stock alto, revisión frecuente'
    else:
        if abc == 'A':
            inventario = 'Stock estratégico, monitoreo diario'
        else:
            inventario = 'Evaluar bajo pedido'

    matriz_decision.append({
        'Grupo': grupo,
        'Cantidad': cant,
        'Prioridad': prioridad,
        'Método_Pronóstico': metodo,
        'Estrategia_Inventario': inventario
    })

df_decision = pd.DataFrame(matriz_decision)
print("\n" + df_decision.to_string(index=False))

# --- Paso 13: Resumen ejecutivo ---
print("\n" + "="*80)
print("📊 RESUMEN EJECUTIVO")
print("="*80)

total_continua = len(df_master[df_master['Tipo_Demanda'] == 'Continua'])
total_intermitente = len(df_master[df_master['Tipo_Demanda'] == 'Intermitente'])

print(f"""
Total de items analizados: {len(df_master)}

CLASIFICACIÓN ABC:
  • Clase A: {len(df_master[df_master['ABC'] == 'A'])} items ({len(df_master[df_master['ABC'] == 'A'])/len(df_master)*100:.1f}%)
  • Clase B: {len(df_master[df_master['ABC'] == 'B'])} items ({len(df_master[df_master['ABC'] == 'B'])/len(df_master)*100:.1f}%)
  • Clase C: {len(df_master[df_master['ABC'] == 'C'])} items ({len(df_master[df_master['ABC'] == 'C'])/len(df_master)*100:.1f}%)

CLASIFICACIÓN XYZ:
  • Clase X: {len(df_master[df_master['XYZ'] == 'X'])} items ({len(df_master[df_master['XYZ'] == 'X'])/len(df_master)*100:.1f}%)
  • Clase Y: {len(df_master[df_master['XYZ'] == 'Y'])} items ({len(df_master[df_master['XYZ'] == 'Y'])/len(df_master)*100:.1f}%)
  • Clase Z: {len(df_master[df_master['XYZ'] == 'Z'])} items ({len(df_master[df_master['XYZ'] == 'Z'])/len(df_master)*100:.1f}%)

TIPO DE DEMANDA:
  • Continua: {total_continua} items ({total_continua/len(df_master)*100:.1f}%)
  • Intermitente: {total_intermitente} items ({total_intermitente/len(df_master)*100:.1f}%)
""")

if total_intermitente > 0:
    suaves_count = len(df_master[df_master['CV2_Clasificacion'] == 'Suave (Forecasteable)'])
    erraticas_count = len(df_master[df_master['CV2_Clasificacion'] == 'Errática (Difícil)'])

    print(f"""
ITEMS INTERMITENTES (Detalle):
  • Suaves (CV² < 0.49): {suaves_count} items ({suaves_count/total_intermitente*100:.1f}% de intermitentes)
    → Forecasteables con Croston/SBA
  • Erráticas (CV² >= 0.49): {erraticas_count} items ({erraticas_count/total_intermitente*100:.1f}% de intermitentes)
    → Muy difíciles, requieren métodos especiales
""")

# Items críticos (A-Intermitente)
criticos = df_master[(df_master['ABC'] == 'A') & (df_master['Tipo_Demanda'] == 'Intermitente')]
if len(criticos) > 0:
    print(f"""
⚠️  ITEMS CRÍTICOS (A-Intermitente): {len(criticos)} items
    → Requieren ATENCIÓN ESPECIAL
    → Alta importancia + Alta complejidad
""")
    print("   Productos críticos:")
    for producto in criticos['Producto'].values[:10]:
        print(f"   • {producto}")
    if len(criticos) > 10:
        print(f"   ... y {len(criticos)-10} más")

print("\n" + "="*80)
print("✅ ANÁLISIS COMPLETADO")
print("="*80)

# Retornar DataFrames principales
print("\n💡 DataFrames generados en memoria:")
print("   • df_master: Clasificación completa de todos los items")
print("   • resumen_simple: Resumen por grupo ABC-XYZ-Tipo")
print("   • tabla_pct: Tabla con porcentajes")
print("   • df_decision: Matriz de decisión con estrategias")


CLASIFICACIÓN COMPLETA: ABC-XYZ-ADI-CV²

✅ Periodo: 2023-01-01 00:00:00 hasta 2025-10-01 00:00:00 (34 períodos)
📊 Productos analizados: 30

PASO 1: CLASIFICACIÓN ABC

📊 Clasificación ABC (80-15-5):
A    22
B     4
C     4
Name: count, dtype: int64

PASO 2: CLASIFICACIÓN XYZ (Coeficiente de Variación)

📊 Clasificación XYZ:
X    10
Y    10
Z    10
Name: count, dtype: int64
   P33 = 0.5661
   P67 = 0.6377

PASO 3: CÁLCULO DE ADI (Average Demand Interval) para items Z

📊 Tipo de Demanda:
Tipo_Demanda
Continua        27
Intermitente     3
Name: count, dtype: int64

PASO 4: CÁLCULO DE CV² para items INTERMITENTES

📖 Clasificación según CV² (Syntetos et al.):
   • CV² < 0.49: Suave (Smooth) - Forecasteable con Croston/SBA
   • CV² >= 0.49: Errática (Erratic) - Muy difícil de forecastear


📊 Total items intermitentes: 3

   Estadísticas de CV²:
   • Media: 2.3021
   • Mediana: 2.2581
   • Mínimo: 1.9650
   • Máximo: 2.6831

📊 Clasificación por CV²:
CV2_Clasificacion
Errática (Difícil)    3
Nam